In [14]:
import pandas as pd
import time
import gc


df = pd.read_csv('../data/fines.csv')

def measure_time(func, *args, **kwargs):

    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return elapsed, result


In [15]:

%%timeit
def calculate_loop(df):
    results = []
    for i in range(len(df)):
        row = df.iloc[i]
        results.append(row['Fines'] / row['Refund'] * row['Year'])
    return results
df['Calculated'] = calculate_loop(df)


19.2 ms ± 129 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%%timeit
def calculate_iterrows(df):
    return [row['Fines'] / row['Refund'] * row['Year'] for _, row in df.iterrows()]
df['Calculated'] = calculate_iterrows(df)


16.2 ms ± 35.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [17]:
%%timeit
df['Calculated'] = df.apply(lambda row: row['Fines'] / row['Refund'] * row['Year'], axis=1)

4.98 ms ± 4.24 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
%%timeit
df['Calculated'] = df['Fines'] / df['Refund'] * df['Year']

82 μs ± 315 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [19]:
%%timeit
df['Calculated'] = df['Fines'].values / df['Refund'].values * df['Year'].values

42.8 μs ± 77.5 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [20]:
%%timeit
row = df[df['CarNumber'] == 'O136HO197RUS']



129 μs ± 1.13 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [21]:
%%timeit
# С индексацией
df_indexed = df.set_index('CarNumber')

row = df_indexed.loc['O136HO197RUS']

277 μs ± 6.13 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [22]:
print("До оптимизации:")
df.info(memory_usage='deep')


df_opt = df.copy()

float_cols = df_opt.select_dtypes(include='float').columns
df_opt[float_cols] = df_opt[float_cols].apply(pd.to_numeric, downcast='float')


int_cols = df_opt.select_dtypes(include='integer').columns
df_opt[int_cols] = df_opt[int_cols].apply(pd.to_numeric, downcast='integer')

print("\nПосле оптимизации:")
df_opt.info(memory_usage='deep')

До оптимизации:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CarNumber   930 non-null    object 
 1   Refund      930 non-null    int64  
 2   Fines       930 non-null    float64
 3   Make        930 non-null    object 
 4   Model       919 non-null    object 
 5   Year        930 non-null    int64  
 6   Calculated  930 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 182.1 KB

После оптимизации:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CarNumber   930 non-null    object 
 1   Refund      930 non-null    int8   
 2   Fines       930 non-null    float32
 3   Make        930 non-null    object 
 4   Model       919 non-null    object 
 5   Year        930 non-null    int1

In [23]:

obj_cols = df_opt.select_dtypes(include='object').columns
df_opt[obj_cols] = df_opt[obj_cols].astype('category')

print("\nПосле категоризации:")
df_opt.info(memory_usage='deep')


После категоризации:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   CarNumber   930 non-null    category
 1   Refund      930 non-null    int8    
 2   Fines       930 non-null    float32 
 3   Make        930 non-null    category
 4   Model       919 non-null    category
 5   Year        930 non-null    int16   
 6   Calculated  930 non-null    float64 
dtypes: category(3), float32(1), float64(1), int16(1), int8(1)
memory usage: 67.0 KB


In [24]:

%reset_selective df

gc.collect()

1042